# 04 — Sequence Distance Features

This notebook converts symbolic patient sequences into numerical features based on their similarity to mined discriminative patterns.

The purpose is to allow partial or near matches instead of requiring exact pattern matches.

Output:
- Distance/similarity feature matrix
- Feature metadata


In [ ]:
from pathlib import Path
import pickle
import ast
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SEQUENCE_DIR = PROJECT_ROOT / "outputs" / "sequences"
PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"
FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

with open(SEQUENCE_DIR / "positive_sequences.pkl", "rb") as f:
    positive_sequences = pickle.load(f)

with open(SEQUENCE_DIR / "negative_sequences.pkl", "rb") as f:
    negative_sequences = pickle.load(f)

patterns_df = pd.read_csv(PATTERN_DIR / "discriminative_patterns.csv")

if not patterns_df.empty:
    patterns_df["pattern"] = patterns_df["pattern"].apply(ast.literal_eval)

patterns = patterns_df["pattern"].tolist()

print("Patterns:", len(patterns))


## Sequence flattening

For the distance stage, each hourly symbolic itemset is flattened into an ordered token sequence.

Example:

```text
[
    ["HR_HIGH", "MAP_LOW"],
    ["TEMP_HIGH"]
]
```

becomes:

```text
["HR_HIGH", "MAP_LOW", "TEMP_HIGH"]
```

The temporal order is preserved.


In [ ]:
def flatten_sequence(sequence):
    return [item for itemset in sequence for item in itemset]


def levenshtein_distance(a, b):
    # Small, dependency-free implementation for symbolic sequences.
    previous = list(range(len(b) + 1))

    for i, x in enumerate(a, start=1):
        current = [i]

        for j, y in enumerate(b, start=1):
            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (x != y)
            current.append(min(insertion, deletion, substitution))

        previous = current

    return previous[-1]


def normalized_distance(a, b):
    if not a and not b:
        return 0.0

    denominator = max(len(a), len(b), 1)
    return levenshtein_distance(a, b) / denominator


def sequence_features(sequences, patterns):
    matrix = []

    flat_patterns = [list(p) for p in patterns]

    for sequence in sequences:
        flat_sequence = flatten_sequence(sequence)
        row = [
            normalized_distance(flat_sequence, pattern)
            for pattern in flat_patterns
        ]
        matrix.append(row)

    return np.asarray(matrix, dtype=np.float32)


In [ ]:
all_sequences = positive_sequences + negative_sequences
labels = np.array(
    [1] * len(positive_sequences) + [0] * len(negative_sequences),
    dtype=np.int64
)

X_distance = sequence_features(all_sequences, patterns)

np.save(FEATURE_DIR / "distance_features.npy", X_distance)
np.save(FEATURE_DIR / "labels.npy", labels)

feature_names = [f"pattern_{i}" for i in range(len(patterns))]
pd.DataFrame(X_distance, columns=feature_names).to_csv(
    FEATURE_DIR / "distance_features.csv", index=False
)

pd.DataFrame({
    "feature_index": range(len(patterns)),
    "pattern": [str(p) for p in patterns],
}).to_csv(FEATURE_DIR / "feature_pattern_mapping.csv", index=False)

print("Feature matrix shape:", X_distance.shape)
